In [90]:
import pandas as pd
import numpy as np
import sys

import openpyxl
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import load_workbook
from openpyxl.styles import numbers
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# Lecture Fichers

In [91]:
import datetime
str_date = datetime.date.today().strftime("%Y%m") 
path_input = fr"Input_Output\{str_date}\Optimisation\Optimiseur_IA.xlsm"

path_input = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\1_PROD\Codes\Optimiseur_IA.xlsm"

In [92]:
data=pd.read_excel(path_input ,sheet_name='Optim', skiprows=[0],usecols=np.arange(60))
data.columns=[col[0].strip().rstrip() if type(col)==list else col for col in data.columns.str.split(".",n=1)]

df_contraintes=pd.read_excel(path_input, sheet_name='Optim', usecols=[0,1,2,3])
df_contraintes.columns=['type_contrainte','category','min','max']

c:\Users\LIJN\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\LIJN\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\LIJN\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\LIJN\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [93]:
data

,Contraintes linéaires,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,key,lb,ub,...,SP Price Target CIQ,IN / ADD / OUT,ESG,Unnamed: 53,Optim,Poids Ptf,Poids Bench,delta,Geozone,Sector
0,NaN,NaN,NaN,NaN,NaN,NaN,NVIDIA Corporation,"0,3%North America",0.051282,0.053282,...,NaN,IN,0.0,NaN,0.053282,0.050496,0.052282,0.001785,1.0,35.0
1,NaN,NaN,NaN,NaN,NaN,NaN,Apple Inc.,"0,3%North America",0.049300,0.051300,...,NaN,IN,0.0,NaN,0.051300,0.050496,0.050300,0.000196,1.0,35.0
2,Zone geo,Thème,Min,Max,Somme lb,Somme ub,Microsoft Corporation,"0,3%North America",0.041224,0.043224,...,NaN,IN,0.0,NaN,0.043224,0.050496,0.042224,0.008273,1.0,35.0
3,North America,1,0.774868,0.778868,0.302649,1.097805,"Micron Technology, Inc.","0,3%North America",0.003206,0.030000,...,NaN,IN,0.0,NaN,0.030000,0.050496,0.003206,0.047291,1.0,35.0
4,West Europe,2,0.158493,0.160493,0.053302,0.26,Kroger Co.,0%North America,0.002000,0.010000,...,NaN,IN,0.0,NaN,0.010000,0.041405,0.000514,0.040891,1.0,34.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1468,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1469,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
constraints_list=['Zone geo', 'Secteur']
dico={ 
    'Zone geo':'Geozone',
    'Secteur':'Sector',
    }

In [95]:
def define_constraints(df_contraintes,df,constraints_list,dico):
    """"
        Permet de définir les contraintes linéaires à utiliser dans l'optimiseur
    """

    linear_constraints={}
    i=0
    boolmask=df_contraintes['type_contrainte'].isin(constraints_list)
    # contrainte thematique
    while i<len(df_contraintes):
        if boolmask.iloc[i]:
            j=i+1
            while j<len(boolmask) and  boolmask.iloc[j]==False:
                j+=1
            linear_constraints[df_contraintes.iloc[i,0]]=df_contraintes.iloc[i+1:j,:4].dropna()
            i=j
        else:
            i+=1


    linear_constraints_matrix = {}
    for col in constraints_list:
            n = int(pd.to_numeric(linear_constraints[col]['category'], errors='coerce').max())
            Matrix=np.zeros((len(data[dico[col]].dropna()),n))
            for i in range(len(Matrix)):
                k=int(data[dico[col]].dropna().iloc[i])
                Matrix[i,k-1]=1
            linear_constraints_matrix[col]=pd.DataFrame(Matrix,columns=['col'+str(i) for i in range(n)])

    lb = df.iloc[1:, 8].dropna().values.astype(float).reshape(-1, 1)
    ub = df.iloc[1:, 9].dropna().values.astype(float).reshape(-1, 1)

    return linear_constraints, linear_constraints_matrix, lb, ub


In [96]:
linear_constraints, linear_constraints_matrix, lb, ub = define_constraints(df_contraintes,data,constraints_list,dico)

Cov/Var Matrix Computation

In [97]:
# GET ISIN-SEDOL
isin_funds = pd.read_excel(path_input, sheet_name='Pos', usecols=[1, 2])
sedol_to_isin = dict(zip(isin_funds['Company SEDOL'], isin_funds['ISIN']))

returns=pd.read_parquet(r"\\GROUPE-UFG.COM\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_RETURNS\returns.parquet")
returns.reset_index(inplace=True)
returns=returns.rename(columns={'index':'Date'})
returns.rename(columns=sedol_to_isin, inplace=True)
returns = returns[[col for col in returns.columns if col in sedol_to_isin.values() or col == 'Date']]
max_date = returns['Date'].max()

# Filtrer les dates pour ne garder que celles dans la plage d'une année à partir de la date maximale
one_year = max_date - pd.DateOffset(years=1)
returns = returns[returns['Date'] >= one_year]
returns = returns.set_index('Date')

In [108]:
returns

,HK0016000132,HK0012000102,HK0083000502,SE0015811963,BMG4587L1090,HK0011000095,BE0003797140,US7443201022,HK0006000050,JP3933800009,...,CA5503711080,CA76131D1033,CA96467A2002,CH1430134226,JP3236330001,US02156V1098,JP3435350008,JP3379550001,US77311W1018,US74743L1008
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-16,-0.010320,-0.015929,-0.009213,0.006285,-0.014849,-0.014283,-0.003828,-0.000636,0.000233,-0.012264,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2024-12-17,-0.009998,-0.013079,-0.018805,-0.007869,-0.019253,-0.001815,-0.001537,-0.013510,-0.004511,0.016735,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2024-12-18,0.001952,0.011020,-0.007680,0.002580,0.002578,0.002630,0.003079,-0.033683,0.016459,-0.011987,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2024-12-19,-0.013653,-0.024633,-0.005556,-0.015460,-0.002479,-0.011557,-0.012279,0.016709,-0.006830,-0.031013,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2024-12-20,0.002054,-0.004228,-0.008661,-0.002501,-0.011032,-0.009138,0.003108,0.018525,0.019585,-0.004594,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-10,0.006239,-0.007990,-0.013323,0.022082,-0.002635,0.000237,-0.006114,0.026431,0.014659,-0.007925,...,0.012101,-0.003651,-0.000188,0.017924,-0.039476,-0.033122,0.015643,0.032209,0.012034,0.011914
2025-12-11,-0.014926,0.007475,0.010073,-0.011281,0.044477,-0.008492,0.004101,0.009978,-0.002112,-0.005768,...,0.020436,-0.014120,0.001241,0.022905,0.013374,0.013842,-0.014030,-0.048195,-0.003605,-0.025864
2025-12-12,0.036771,0.010862,0.014379,-0.001236,0.036263,0.001254,0.001362,0.000806,0.024360,0.003992,...,-0.002209,0.007547,0.005944,0.000331,0.025964,-0.149925,0.028870,0.038309,-0.032051,-0.067637


In [99]:
sigma=returns.cov().to_numpy()
sigma = sigma * 252
w_bench = data['Poids Bench'].dropna().to_numpy(dtype=float)
w0 = data['Poids Ptf'].dropna().to_numpy(dtype=float)
score_ml = data['Score ML'].dropna().iloc[0:].to_numpy(dtype=float)
lower_bound=data['lb'].dropna()
upper_bound=data['ub'].dropna()

In [100]:
def compute_TE(w0,w_bench,sigma):
    diff= w0 - w_bench
    te = float(diff @ sigma @ diff)
    te = np.sqrt(te)
    return te

TE = compute_TE(w0,w_bench,sigma)
print(f"Tracking Error for initial PTF is {TE}")

Tracking Error for initial PTF is 0.03136274387010716


In [101]:
linear_constraints

{'Zone geo':   type_contrainte category       min       max
 4   North America        1  0.774868  0.778868
 5     West Europe        2  0.158493  0.160493
 6          Others        3  0.062639  0.064639,
 'Secteur':                               type_contrainte category       min       max
 10                   Auto & Parts_West Europe        1  0.001702  0.003702
 11                          Banks_West Europe        2  0.023662  0.025662
 12                Basic Resources_West Europe        3  0.002265  0.004265
 13                      Chemicals_West Europe        4  0.002041  0.004041
 14                   Construction_West Europe        5  0.004364  0.006364
 15                         Energy_West Europe        6  0.007004  0.009004
 16             Financial Services_West Europe        7  0.005885  0.007885
 17       Food, Beverage & Tobacco_West Europe        8  0.003387  0.005387
 18                    Health Care_West Europe        9  0.021261  0.023261
 19    Industrial Goods 

In [102]:
sigma

array([[ 0.05975244,  0.04590487,  0.03089914, ...,  0.00870209,
         0.0010752 , -0.00306073],
       [ 0.04590487,  0.06810214,  0.02827515, ...,  0.00160459,
        -0.0012589 , -0.00071327],
       [ 0.03089914,  0.02827515,  0.03487628, ...,  0.0035844 ,
         0.00176194, -0.00105166],
       ...,
       [ 0.00870209,  0.00160459,  0.0035844 , ...,  0.04928945,
        -0.01165206, -0.00027217],
       [ 0.0010752 , -0.0012589 ,  0.00176194, ..., -0.01165206,
         0.03300526,  0.00126756],
       [-0.00306073, -0.00071327, -0.00105166, ..., -0.00027217,
         0.00126756,  0.01221245]])

In [103]:
linear_constraints_matrix

{'Zone geo':       col0  col1  col2
 0      1.0   0.0   0.0
 1      1.0   0.0   0.0
 2      1.0   0.0   0.0
 3      1.0   0.0   0.0
 4      1.0   0.0   0.0
 ...    ...   ...   ...
 1305   1.0   0.0   0.0
 1306   0.0   1.0   0.0
 1307   0.0   0.0   1.0
 1308   0.0   0.0   1.0
 1309   0.0   0.0   1.0
 
 [1310 rows x 3 columns],
 'Secteur':       col0  col1  col2  col3  col4  col5  col6  col7  col8  col9  ...  col47  \
 0      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 1      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 2      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 3      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 4      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 ...    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...    ...   
 1305   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
 1306   0.0  

# Optimiseur

In [104]:
import numpy as np
import cvxpy as cp

def optimize_tracking_error(
    sigma,
    w_bench,
    lower_bound=None,
    upper_bound=None,
    solver=cp.ECOS,
    verbose=False
    ):
    import cvxpy as cp
    sigma = np.asarray(sigma, dtype=float)
    w_bench = np.asarray(w_bench, dtype=float)
    
    n = len(w_bench)
    
    if sigma.shape != (n, n):
        raise ValueError("sigma doit être une matrice carrée de taille (n, n) correspondant à w_bench.")
    
    # Variable de décision : poids du portefeuille
    w = cp.Variable(n)
    # Objectif : minimiser la variance de la tracking error (TE^2)
    
    # diff = w - w_bench
    # objective = cp.Minimize(cp.quad_form(diff, sigma))
    # z = w - w_bench
    # objective =  cp.Minimize(cp.sum_squares(sigma @ z))

    # Min Turnover
    objective =  cp.Minimize( cp.norm(w - w0, 2))

    # Max Score ML
    # objective =  cp.Maximize( w @ score_ml)

    # Contraintes
    constraints = []

    constraints.append(cp.sum(w) == 1)
    constraints.append(w>=0)

    constraints.append(w >= lower_bound)
    constraints.append(w <= upper_bound)


    active_contrainte = [
            'Zone geo',
            'Secteur'
    ]
    eps = 10e-6
    for col in active_contrainte:

        if col not in linear_constraints or col not in linear_constraints_matrix:
            raise ValueError(f"Contrainte manquante {col} : "
                                f"len(A)={len(A1d)}, n_assets={n}")


        lc = linear_constraints[col]
        A_mat = linear_constraints_matrix[col]
        print(A_mat)
        if isinstance(A_mat, pd.DataFrame):
            A = A_mat.values.astype(float)   

        else:

            A1d = np.asarray(A_mat, dtype=float).flatten()
            if A1d.shape[0] != n:
                raise ValueError(f"Dimension incohérente pour la contrainte {col} : "
                                f"len(A)={len(A1d)}, n_assets={n}")
            A = A1d.reshape(-1, 1)                   # (n_assets, 1)

        if A.shape[0] != n:
            raise ValueError(f"Première dimension de A pour {col} ne vaut pas n_assets : "
                            f"A.shape={A.shape}, n_assets={n}")


        if isinstance(lc, pd.DataFrame):
            # colonnes : [..., 'min', 'max']
            m = lc.iloc[:, 2].to_numpy(dtype=float)  # min
            M = lc.iloc[:, 3].to_numpy(dtype=float)  # max

            if A.shape[1] != len(m):
                raise ValueError(f"Incohérence dimensions pour {col} : "
                                f"A.shape[1]={A.shape[1]}, len(m)={len(m)}")


            constraints.append(A.T @ w >= m)

            constraints.append(A.T @ w <= M + eps)

        else:

            bounds_arr = np.asarray(lc, dtype=float).flatten()
            if bounds_arr.size < 4:
                raise ValueError(f"Format inattendu de linear_constraints['{col}'] : {bounds_arr}")

            m = float(bounds_arr[2])
            M = float(bounds_arr[3])


            a_vec = A[:, 0]


            constraints.append(a_vec @ w >= m)


            if np.isfinite(M):
                constraints.append(a_vec @ w <= M + eps)

            print(f"Contrainte target '{col}' : min={m}, max={M}")

    prob = cp.Problem(objective, constraints)
    print("is_dcp:", prob.is_dcp())
    prob.solve(solver=cp.CVXOPT, verbose=True)
    
    result = {  
        "status": prob.status,
        "objective_value": prob.value,
    }
    print(result)
    if w.value is None or prob.status not in ["optimal", "optimal_inaccurate"]:
        # Problème non résolu correctement
        
        return None, None, result
    
    w_opt = np.asarray(w.value).flatten()
    
    # Calcul de la tracking error ex ante (écart-type)
    diff_opt = w_opt - w_bench
    print(diff_opt)
    te_var = float(diff_opt @ sigma @ diff_opt)
    TE_opt = np.sqrt(max(te_var, 0.0))
    

    result["TE_opt"] = TE_opt
    
    return w_opt, TE_opt, result

In [105]:
import cvxpy

w_opt, TE_opt, info = optimize_tracking_error(
    sigma=sigma,
    w_bench=w_bench,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    solver=cvxpy.CVXOPT,
    verbose=False
)

(CVXPY) déc. 17 10:47:49 : Your problem has 1310 variables, 4051 constraints, and 0 parameters.
(CVXPY) déc. 17 10:47:49 : It is compliant with the following grammars: DCP, DQCP
(CVXPY) déc. 17 10:47:49 : (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) déc. 17 10:47:49 : CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) déc. 17 10:47:49 : Your problem is compiled with the CPP canonicalization backend.
(CVXPY) déc. 17 10:47:49 : Compiling problem (target solver=CVXOPT).
(CVXPY) déc. 17 10:47:49 : Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> CVXOPT
(CVXPY) déc. 17 10:47:49 : Applying reduction Dcp2Cone
(CVXPY) déc. 17 10:47:49 : Applying reduction CvxAttr2Constr
(CVXPY) déc. 17 10:47:49 : Applying reduction ConeMatrixStuffing
(CVXPY) déc. 17 10:47:49 : Applying reduction CVXOPT
(CVXPY) déc. 17 10:47:49 : Finished problem compilation (took 2.01

      col0  col1  col2
0      1.0   0.0   0.0
1      1.0   0.0   0.0
2      1.0   0.0   0.0
3      1.0   0.0   0.0
4      1.0   0.0   0.0
...    ...   ...   ...
1305   1.0   0.0   0.0
1306   0.0   1.0   0.0
1307   0.0   0.0   1.0
1308   0.0   0.0   1.0
1309   0.0   0.0   1.0

[1310 rows x 3 columns]
      col0  col1  col2  col3  col4  col5  col6  col7  col8  col9  ...  col47  \
0      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
1      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
2      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
3      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
4      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
...    ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...    ...   
1305   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...    0.0   
1306   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0 

(CVXPY) déc. 17 10:47:59 : Problem status: optimal
(CVXPY) déc. 17 10:47:59 : Optimal value: 6.376e-02
(CVXPY) déc. 17 10:47:59 : Compilation took 2.016e-02 seconds
(CVXPY) déc. 17 10:47:59 : Solver (including time spent in interface) took 9.742e+00 seconds


19:  6.3757e-02  6.3757e-02  6e-08  2e-08  1e-08  2e-11
Optimal solution found.
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
{'status': 'optimal', 'objective_value': 0.06375736137743644}
[ 1.00000000e-03  1.00000017e-03  1.00000023e-03 ... -1.45815030e-05
 -1.38022781e-05 -5.33580241e-10]


In [106]:
w0 = pd.DataFrame(w0)
w_opt=pd.DataFrame(w_opt)

In [107]:
import pandas as pd
from openpyxl import load_workbook

xlsx_path = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\1_PROD\Codes\x_optim.xlsx"

workbook = load_workbook(xlsx_path)

sheet_name = 'optimisation'
if sheet_name in workbook.sheetnames:
    sheet = workbook[sheet_name]

    for row in sheet.iter_rows():
        for cell in row:
            cell.value = None
    # Sauvegarder le nettoyage
    workbook.save(xlsx_path)

# Réécrire les DataFrames après nettoyage
with pd.ExcelWriter(xlsx_path, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    w0.to_excel(writer, sheet_name=sheet_name, startrow=0, startcol=1, index=False, header=False)
    w_opt.to_excel(writer, sheet_name=sheet_name, startrow=0, startcol=0, index=False, header=False)
    pd.DataFrame([[TE]]).to_excel(writer, sheet_name=sheet_name, startrow=0, startcol=4, index=False, header=False)
    pd.DataFrame([[TE_opt]]).to_excel(writer, sheet_name=sheet_name, startrow=0, startcol=5, index=False, header=False)


In [ ]:
world_ptf_save_path_pickle = fr"Input_Output\\{str_date}\\Output_PTF\\PTF_WORLD.pkl"
world_ptf_save_path_excel = fr"Input_Output\\{str_date}\\Output_PTF\\PTF_WORLD.xlsx"
perf_save_path = fr"Input_Output\\{str_date}\\Output_PTF\\Perf_ESG_WORLD.pkl"

# POIDS REGIONAUX DANS ACWI 
weight_regions_acwi = (
                        screen_agg
                        .groupby('Date')
                        .apply(compute_weights_for_date_acwi)
                        )

# REBALANCE 100%
weight_regions_acwi = weight_regions_acwi.div(weight_regions_acwi.sum(axis=1), axis=0)

# POIDS REGIONAUX DANS MSCI WORLD after 2017
weight_regions_world = (
                        screen_agg[screen_agg['Date']>="2017-01-01"]
                        .groupby('Date')
                        .apply(compute_weights_for_date_world)
                        )

# REBALANCE 100%
weight_regions_world = weight_regions_world.div(weight_regions_world.sum(axis=1), axis=0)


# Replace weight after 2017 with MSCI WORLD data
weight_regions_acwi.loc[weight_regions_world.index, ['US weight', 'EU weight', 'Other weight']] = weight_regions_world[['US weight', 'EU weight', 'Other weight']]
weight_regions_bench = weight_regions_acwi.copy(deep=True)

# Align date of regional weight with date of ptf generated
weight_regions_bench.index = weight_regions_bench.index + pd.offsets.MonthBegin(1) # Both regional weight or ptf will have the first day of month as Date


# SOUS PONDERER OTHER DE 0.8
weight_regions_bench = adjust_weights_OTHER(weight_regions_bench, ratio_other=0.8)


us_adjusted = adjust_single_portfolio_weights(PTF_US, weight_regions_bench.reset_index(), 'US')
eu_adjusted = adjust_single_portfolio_weights(PTF_EU, weight_regions_bench.reset_index(), 'EU')
other_adjusted = adjust_single_portfolio_weights(PTF_OTHER, weight_regions_bench.reset_index(), 'Other')

ptf_world = pd.concat([us_adjusted, eu_adjusted, other_adjusted], axis=0)
ptf_world = ptf_world[ptf_world['Date'] >= "2010-03-01"]

ptf_world = ptf_world.drop_duplicates(subset=["Date", "ISIN"])

# Rebalancing
ptf_world['Weight'] = (
                            ptf_world.groupby(["Date"])['Weight']
                            .transform(lambda w: w / w.sum())
                            )



In [ ]:
from Codes.ML_SUIVI import *
ptf_world = ptf_world_bk.copy(deep=True)
date_sec_list = ptf_world["Date"].max()
ptf_world_bloom = ptf_world[ptf_world["Date"] == date_sec_list]

# Consturction du bench
cols = ['Dividend Avg Percentile', 'Value Avg Percentile', 'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile', 'Growth Avg Percentile', "Score ML", "Company SEDOL"]
df_screen = add_reco_analyst_multi_facteur(path_screen, path_ciq, date_sec_list, bench="MSCI WORLD", optional_cols=cols)

from Codes.BacktestEngine import merge_ticker_secondaire
df_screen = merge_ticker_secondaire(df_screen)

print("Adding Exclusion Reasons......")
list_exclu_total = get_list_exclusion("WORLD")
df_screen = df_screen.merge(list_exclu_total[['ISIN', 'Raison Exclusion']], how="left", on='ISIN')

# keep only these two values; change everything else to "Others"
keep = ['North America', 'West Europe']
df_screen.loc[~df_screen['Exchange Country Region'].isin(keep), 'Exchange Country Region'] = 'Others'



# Concatate PTF
df_screen = df_screen.merge(ptf_world_bloom[["ISIN", "Weight", "Raison Repechage", "Date"]],
                how="left",
                left_on="ISIN",
                right_on="ISIN"
                )

df_screen['Date'] = date_sec_list
df_screen = df_screen.drop_duplicates(subset="ISIN")

# Treat blanks and NaNs as empty
rep = df_screen['Raison Repechage'].fillna('').str.strip()
exc = df_screen['Raison Exclusion'].fillna('').str.strip()
# 1) If Raison Repechage is not empty -> empty Raison Exclusion
df_screen.loc[rep.ne(''), 'Raison Exclusion'] = ''
# 2) If both empty -> "Not Covered" in Raison Exclusion
df_screen.loc[(rep.eq('')) & (exc.eq('')), 'Raison Exclusion'] = 'Not Covered'


df_screen['Weight'] = (
                        df_screen['Weight']
                        .transform(lambda w: w / w.sum())
                        )
df_screen[f'Weight in MSCI WORLD'] = (
                        df_screen['Weight in MSCI WORLD']
                        .transform(lambda w: w / w.sum())
                        )

df_screen['Score ML'] = df_screen.groupby(['Exchange Country Region', "ICB19 Supersector"])['Score ML'].rank(  
                                                                                pct=True,  
                                                                                ascending=True  
                                                                                ) * 10
df_screen['Score ML'] = df_screen['Score ML'].fillna(-1)

cols_ordered = ["Date",
    'ISIN', 'Company SEDOL',
    'Name', 'ICB19 Supersector',
    'Exchange Country Region', 'Exchange Country Name',
    'Weight', 'Weight in MSCI WORLD',
    'Reco Analyst', 'Score ML',
    'Raison Exclusion', 'Raison Repechage',
    'Dividend Avg Percentile', 'Value Avg Percentile', 'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile', 'Growth Avg Percentile',
    'Multi Avg Percentile',
    'EPS Growth FY1', 'ROE avg FY0', 'Oper Margin', 'PE LTM',
    'Earns Yield FY0', 'DVD Yield FY0'
]

df_screen = df_screen[cols_ordered]

df_screen.to_excel(fr"Input_Output\{str_date}\Output_PTF\PTF_WORLD_non_optim.xlsx", index=False)